# Brouillon Julien — modèles, seuil fixé par les coûts, résultat en euros

Première version avec des features simples. Le **test n'est pas ouvert** ici : tout se compare sur la validation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

users = pd.read_csv('rev-users.csv', parse_dates=['created_date'])
tx = pd.read_csv('rev-transactions.csv', parse_dates=['created_date'])

# Règles fixées en partie 1
WINDOW = pd.Timedelta(days=60)
MIN_TENURE = pd.Timedelta(days=60)
CUTOFFS = {'fit': pd.Timestamp('2019-01-15'),
           'validation': pd.Timestamp('2019-02-15'),
           'test': pd.Timestamp('2019-03-15')}
X_OFFRE, Y_CLIENT, BUDGET = 5, 16.8, 500   # € par offre, € par client retenu, offres/mois

## 1. Un tableau par coupure : une ligne par client actif

Features calculées **uniquement avant** la coupure ; label calculé **uniquement après**.

In [ ]:
def build_simple(cutoff):
    passe = tx[tx.created_date < cutoff]
    futur = tx[(tx.created_date >= cutoff) & (tx.created_date < cutoff + WINDOW)]

    # clients actifs : au moins une transaction dans les 60 jours avant la coupure
    actifs = passe[passe.created_date >= cutoff - WINDOW].user_id.unique()
    d = pd.DataFrame({'user_id': actifs})

    # ancienneté, et exclusion des clients de moins de 60 jours
    signup = users.set_index('user_id').created_date
    d['anciennete_j'] = (cutoff - d.user_id.map(signup)).dt.days
    d = d[d.anciennete_j >= MIN_TENURE.days]

    # récence : jours depuis la dernière transaction
    derniere = passe.groupby('user_id').created_date.max()
    d['recence_j'] = (cutoff - d.user_id.map(derniere)).dt.days

    # volume récent et tendance
    for jours in [30, 90]:
        fenetre = passe[passe.created_date >= cutoff - pd.Timedelta(days=jours)]
        d[f'nb_tx_{jours}j'] = d.user_id.map(fenetre.groupby('user_id').size()).fillna(0)
    d['tendance'] = d.nb_tx_30j / (d.nb_tx_90j / 3 + 1)   # >1 : accélère, <1 : ralentit

    # label : aucune transaction dans les 60 jours après la coupure
    d['churn'] = (~d.user_id.isin(futur.user_id)).astype(int)
    return d.reset_index(drop=True)

fit = build_simple(CUTOFFS['fit'])
val = build_simple(CUTOFFS['validation'])
print(f'fit : {len(fit)} clients, churn {fit.churn.mean():.1%}')
print(f'validation : {len(val)} clients, churn {val.churn.mean():.1%}')

## 2. Baseline, régression logistique, arbres boostés

**Baseline (une ligne) :** plus un client n'a rien fait depuis longtemps, plus il est à risque → on le classe par récence.

In [ ]:
FEATURES = ['anciennete_j', 'recence_j', 'nb_tx_30j', 'nb_tx_90j', 'tendance']

scores = {}
scores['baseline (récence)'] = val.recence_j

logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logit.fit(fit[FEATURES], fit.churn)
scores['régression logistique'] = logit.predict_proba(val[FEATURES])[:, 1]

trees = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, random_state=0)
trees.fit(fit[FEATURES], fit.churn)
scores['arbres boostés'] = trees.predict_proba(val[FEATURES])[:, 1]

## 3. Résultat : PR-AUC et euros, avec 500 offres

Pour chaque méthode : on classe les clients de validation par risque, on prend les 500 premiers, on compte les vrais churners touchés.

Gain = churners touchés × Z × Y − 500 × X. **Z est inconnu** : on montre plusieurs valeurs.

In [ ]:
rows = []
for nom, s in scores.items():
    top = val.assign(score=s).nlargest(BUDGET, 'score')
    touches = top.churn.sum()
    row = {'méthode': nom,
           'PR-AUC': round(average_precision_score(val.churn, s), 3),
           'churners dans le top 500': int(touches)}
    for z in [0.30, 0.50]:
        row[f'gain € (Z={z:.0%})'] = round(touches * z * Y_CLIENT - BUDGET * X_OFFRE)
    rows.append(row)

# référence : 500 clients tirés au hasard
hasard = val.churn.mean() * BUDGET
rows.append({'méthode': 'hasard', 'PR-AUC': round(val.churn.mean(), 3),
             'churners dans le top 500': round(hasard),
             'gain € (Z=30%)': round(hasard * 0.30 * Y_CLIENT - BUDGET * X_OFFRE),
             'gain € (Z=50%)': round(hasard * 0.50 * Y_CLIENT - BUDGET * X_OFFRE)})
pd.DataFrame(rows)